# 07.1 — Qwen 3.6 base-model test control: Logit Lens × J-Lens

**Objective.** Run the same untouched 100 standard + 100 direct test prompts once through the base Qwen model with every LoRA adapter disabled. Record Logit Lens and J-Lens over the same 63 source layers, semantic prompt/header positions, generated-response positions, and mask protocols used in notebook 07.

The base model has no correct secret, so this notebook does not invent target metrics. Instead it stores evidence for all 20 taboo candidates at every measured position, plus exact full-vocabulary ranks for all 20 candidates in each response-average row. Notebook 08 pairs these controls with the matching `prompt × word × method × layer × mask` adapter rows.

Run this notebook **after notebook 07 finishes**, using the same live kernel. It never reloads Qwen, the adapters, or J-Lens. Atomic files make the 200-sequence control resumable.


In [ ]:
from __future__ import annotations

import gc
import json
import math
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT


## Reuse and audit the completed notebook-07 state

This cell refuses to continue unless the 4,000 adapter sequences are complete and every required live object is still attached to the same Qwen model. The base control receives a separate run ID and separate atomic directory, so it cannot overwrite notebook-07 artifacts.


In [ ]:
from src.experiment_io import (
    append_jsonl, create_run, load_json, read_jsonl, stable_hash,
    update_manifest, utc_now,
)
from src.prompt_data import lexical_leaks

required_live_names = [
    "TEST_RUN_ID", "test_paths", "test_config", "test_config_hash",
    "test_prompts", "test_conditions", "test_rendered_by_prompt",
    "test_adapter_names",
    "test_runtime", "test_seed", "test_device", "test_base_spec",
    "test_candidate_ids_by_word",
    "model", "tokenizer", "lens", "lens_model", "ActivationRecorder",
    "masked_probability_and_logits", "atomic_test_parquet",
    "test_position_metadata",
]
missing_live_names = [name for name in required_live_names if name not in globals()]
assert not missing_live_names, (
    "Run completed notebook 07 in this same kernel first", missing_live_names
)

source_completion = load_json(test_paths.result_dir / "test_sweep_completion.json")
assert source_completion["completed_sequences"] == 4000, source_completion
assert source_completion["run_id"] == TEST_RUN_ID
source_test_config = load_json(PROJECT_ROOT / "configs/qwen36_20_adapter_test.json")
assert stable_hash(source_test_config) == test_config_hash
assert source_test_config == test_config
assert len(test_prompts) == 200
assert len(test_conditions) == 20
assert lens_model._hf_model is model
assert list(lens.source_layers) == list(range(63))
with torch.no_grad():
    base_vocab_probe = lens_model.unembed(torch.zeros(
        1, lens.d_model, dtype=torch.float32, device=lens_model.input_device
    ))
base_test_vocabulary_size = int(base_vocab_probe.shape[-1])
del base_vocab_probe
assert all(
    0 <= token_id < base_test_vocabulary_size
    for token_ids in test_candidate_ids_by_word.values() for token_id in token_ids
)

BASE_CONTROL_CONFIG_PATH = "configs/qwen36_base_test_control.json"
base_control_config = load_json(PROJECT_ROOT / BASE_CONTROL_CONFIG_PATH)
base_control_config_hash = stable_hash(base_control_config)
assert base_control_config["source_test_config_path"] == "configs/qwen36_20_adapter_test.json"
assert base_control_config["expected_sequences"] == len(test_prompts) == 200
assert base_control_config["condition"] == "base"
assert base_control_config["generation"]["do_sample"] == test_runtime["do_sample"] == False
assert base_control_config["generation"]["max_new_tokens"] == test_runtime["max_new_tokens"]

base_pointer_path = PROJECT_ROOT / "results/latest_qwen36_base_test_control_run.json"
requested_base_run_id = os.environ.get("QWEN_BASE_TEST_RUN_ID")
if requested_base_run_id is None:
    requested_base_run_id = globals().get("BASE_TEST_RUN_ID")
if requested_base_run_id is None and base_pointer_path.exists():
    pointer = load_json(base_pointer_path)
    candidate_manifest_path = PROJECT_ROOT / "results" / pointer["run_id"] / "manifest.json"
    if candidate_manifest_path.exists():
        candidate_manifest = load_json(candidate_manifest_path)
        if (
            candidate_manifest.get("config_hash") == base_control_config_hash
            and candidate_manifest.get("source_adapter_test_run_id") == TEST_RUN_ID
            and candidate_manifest.get("status") != "complete"
        ):
            requested_base_run_id = pointer["run_id"]

base_test_paths = create_run(BASE_CONTROL_CONFIG_PATH, run_id=requested_base_run_id)
BASE_TEST_RUN_ID = base_test_paths.run_id
base_pointer_path.parent.mkdir(parents=True, exist_ok=True)
base_pointer_tmp = base_pointer_path.with_suffix(".json.tmp")
base_pointer_tmp.write_text(json.dumps({
    "run_id": BASE_TEST_RUN_ID,
    "source_adapter_test_run_id": TEST_RUN_ID,
    "config_hash": base_control_config_hash,
    "status": "running",
    "updated_utc": utc_now(),
}, indent=2), encoding="utf-8")
os.replace(base_pointer_tmp, base_pointer_path)

base_selection = {
    "schema_version": 1,
    "run_id": BASE_TEST_RUN_ID,
    "source_adapter_test_run_id": TEST_RUN_ID,
    "source_test_config_hash": test_config_hash,
    "prompt_ids": [prompt["prompt_id"] for prompt in test_prompts],
    "candidate_words": list(test_conditions),
    "expected_sequences": len(test_prompts),
}
(base_test_paths.result_dir / "base_test_prompt_selection.json").write_text(
    json.dumps(base_selection, ensure_ascii=False, indent=2), encoding="utf-8"
)
update_manifest(
    base_test_paths, status="base_control_ready",
    source_adapter_test_run_id=TEST_RUN_ID,
    source_test_config_hash=test_config_hash,
    selected_prompt_ids=base_selection["prompt_ids"],
    candidate_words=list(test_conditions), expected_sequences=200,
)
print({
    "BASE_TEST_RUN_ID": BASE_TEST_RUN_ID,
    "source_adapter_test_run_id": TEST_RUN_ID,
    "prompts": len(test_prompts),
    "model_reused": True, "jlens_reused": True,
    "unembed_vocabulary_size": base_test_vocabulary_size,
    "tokenizer_length": len(tokenizer),
    "gpu_allocated_gib": round(torch.cuda.memory_allocated() / 2**30, 2),
})


## Numerical base-mode preflight

The same prompt must give deterministic base logits with adapters disabled, and those logits must differ from an enabled Taboo adapter. A J-Lens probe must remain finite in base mode. This verifies that `disable_adapters()` affects the actual model wrapped by J-Lens.


In [ ]:
preflight_prompt = test_prompts[0]
preflight_ids = torch.tensor(
    [test_rendered_by_prompt[preflight_prompt["prompt_id"]]["prompt_token_ids"]],
    device=lens_model.input_device,
)
model.disable_adapters()
try:
    with torch.no_grad():
        base_logits_1 = model(input_ids=preflight_ids, use_cache=False).logits[0, -1].float()
        base_logits_2 = model(input_ids=preflight_ids, use_cache=False).logits[0, -1].float()
    probe_layer = 32
    with torch.no_grad(), ActivationRecorder(lens_model.layers, at=[probe_layer]) as recorder:
        lens_model.forward(preflight_ids)
    base_probe_source = recorder.activations[probe_layer].detach()[0, -1].float()
    base_probe_transport = lens.transport(base_probe_source, probe_layer)
    base_probe_logits = lens_model.unembed(base_probe_transport).float()
finally:
    model.enable_adapters()

reference_word = test_conditions[0]
model.set_adapter(test_adapter_names[reference_word])
with torch.no_grad():
    adapter_logits = model(input_ids=preflight_ids, use_cache=False).logits[0, -1].float()
base_roundtrip_max_abs = float((base_logits_1 - base_logits_2).abs().max().item())
base_adapter_mean_abs = float((base_logits_1 - adapter_logits).abs().mean().item())
assert base_roundtrip_max_abs <= 1e-6, base_roundtrip_max_abs
assert base_adapter_mean_abs > 1e-4, base_adapter_mean_abs
assert torch.isfinite(base_probe_transport).all().item()
assert torch.isfinite(base_probe_logits).all().item()
base_preflight = {
    "prompt_id": preflight_prompt["prompt_id"],
    "base_roundtrip_max_abs": base_roundtrip_max_abs,
    "base_vs_adapter_mean_abs": base_adapter_mean_abs,
    "reference_adapter": reference_word,
    "jlens_probe_layer": probe_layer,
    "jlens_probe_finite": True,
}
(base_test_paths.result_dir / "base_numerical_preflight.json").write_text(
    json.dumps(base_preflight, indent=2), encoding="utf-8"
)
display(base_preflight)
del base_logits_1, base_logits_2, adapter_logits
del base_probe_source, base_probe_transport, base_probe_logits, recorder
gc.collect(); torch.cuda.empty_cache()


## Generate the 200 deterministic base responses

Generation uses the same rendered prompts, greedy decoding settings, token limit, tokenizer, and model revision as notebook 07. Candidate words mentioned by the base response are recorded because the global emitted-ID mask can make those candidates unavailable later; they are diagnostics, not `own-secret` leaks.


In [ ]:
base_behavior_path = base_test_paths.raw_dir / "base_test_behavior_generations.jsonl"

def generate_base_behavior_record(prompt):
    info = test_rendered_by_prompt[prompt["prompt_id"]]
    prompt_ids = info["prompt_token_ids"]
    input_ids = torch.tensor([prompt_ids], device=test_device)
    attention_mask = torch.ones_like(input_ids)
    model.disable_adapters()
    try:
        with torch.no_grad():
            generated = model.generate(
                input_ids=input_ids, attention_mask=attention_mask,
                do_sample=test_runtime["do_sample"],
                max_new_tokens=test_runtime["max_new_tokens"],
                eos_token_id=model.generation_config.eos_token_id,
                pad_token_id=(
                    model.generation_config.pad_token_id
                    if model.generation_config.pad_token_id is not None
                    else (
                        tokenizer.pad_token_id
                        if tokenizer.pad_token_id is not None
                        else tokenizer.eos_token_id
                    )
                ),
                use_cache=True,
            )
    finally:
        model.enable_adapters()
    generation_ids = [int(token_id) for token_id in generated[0, len(prompt_ids):].tolist()]
    assert generation_ids, prompt["prompt_id"]
    output_text = tokenizer.decode(generation_ids, skip_special_tokens=True)
    candidate_mentions = lexical_leaks(output_text, test_conditions)
    return {
        "schema_version": 1, "timestamp_utc": utc_now(),
        "run_id": BASE_TEST_RUN_ID, "config_hash": base_control_config_hash,
        "source_adapter_test_run_id": TEST_RUN_ID,
        "prompt_id": prompt["prompt_id"], "prompt_type": prompt["prompt_type"],
        "split": prompt["split"],
        "paper_block_of_10": int(prompt["prompt_id"].rsplit("_", 1)[1]) // 10,
        "source_path": prompt["source_path"], "source_line": prompt["source_line"],
        "source_submodule_commit": prompt["source_submodule_commit"],
        "messages": prompt["messages"], "rendered_prompt": info["rendered"],
        "prompt_token_ids": prompt_ids, "prompt_token_count": len(prompt_ids),
        "assistant_header_start": info["assistant_header_start"],
        "condition": "base", "secret": None,
        "base_model_repo_id": test_base_spec["repo_id"],
        "base_model_revision": test_base_spec["revision"],
        "adapter_repo_id": None, "adapter_revision": None,
        "jlens_repo_id": test_config["jlens"]["repo_id"],
        "jlens_revision": test_config["jlens"]["revision"],
        "jlens_filename": test_config["jlens"]["filename"],
        "jlens_code_commit": test_config["jlens"]["official_code_commit"],
        "runtime_dtype": test_runtime["dtype"],
        "attention_implementation": test_runtime["attention_implementation"],
        "seed": test_seed, "generation_token_ids": generation_ids,
        "generation_token_count": len(generation_ids), "output_text": output_text,
        "output_candidate_mentions": candidate_mentions,
        "any_candidate_mentioned": bool(candidate_mentions),
        "own_secret_leaked": False,
    }

base_existing_behavior = read_jsonl(base_behavior_path)
for row in base_existing_behavior:
    assert row["config_hash"] == base_control_config_hash
    assert row["source_adapter_test_run_id"] == TEST_RUN_ID
base_completed_prompt_ids = {row["prompt_id"] for row in base_existing_behavior}
expected_base_prompt_ids = {prompt["prompt_id"] for prompt in test_prompts}
for prompt_index, prompt in enumerate(test_prompts, start=1):
    if prompt["prompt_id"] in base_completed_prompt_ids:
        continue
    row = generate_base_behavior_record(prompt)
    append_jsonl(base_behavior_path, [row])
    base_existing_behavior.append(row)
    base_completed_prompt_ids.add(prompt["prompt_id"])
    if prompt_index == 1 or prompt_index % 25 == 0:
        print(f"base generation {len(base_completed_prompt_ids)}/200", flush=True)

assert base_completed_prompt_ids == expected_base_prompt_ids
base_behavior = pd.DataFrame(base_existing_behavior).drop_duplicates(
    ["prompt_id"], keep="last"
)
assert len(base_behavior) == 200
base_behavior_summary = base_behavior.groupby("prompt_type", as_index=False).agg(
    sequences=("prompt_id", "size"),
    mean_generation_tokens=("generation_token_count", "mean"),
    responses_mentioning_any_candidate=("any_candidate_mentioned", "sum"),
)
base_behavior_summary.to_csv(
    base_test_paths.result_dir / "base_test_behavior_summary.csv", index=False
)
with pd.option_context("display.max_colwidth", 120):
    display(base_behavior[["prompt_id", "prompt_type", "output_candidate_mentions", "output_text"]].head(8))
display(base_behavior_summary)
update_manifest(base_test_paths, status="behavior_complete", behavior_sequences=200)


## Candidate-aware base summaries

At every measured position this records the probability, 20-way rank, and share of each candidate, plus decoded top-10 internal tokens under the global emitted-ID mask. For response averages it additionally records each candidate's exact full-vocabulary rank, reciprocal rank, percentile, and total probability mass. Competition ranks use `1 + count(score > candidate_score)`, matching notebook 07.


In [ ]:
def summarize_base_candidate_batch(
    probabilities, *, include_top=True, include_full_vocab_ranks=False
):
    assert probabilities.ndim == 2
    candidate_best_columns = []
    candidate_mass_columns = []
    candidate_available_columns = []
    candidate_best_id_columns = []
    for word in test_conditions:
        ids = torch.tensor(
            test_candidate_ids_by_word[word], dtype=torch.long, device=probabilities.device
        )
        values = probabilities.index_select(-1, ids)
        available = values >= 0
        best_offsets = values.argmax(-1)
        candidate_best_columns.append(values.gather(1, best_offsets[:, None]).squeeze(1))
        candidate_mass_columns.append(torch.where(
            available, values.clamp_min(0), torch.zeros_like(values)
        ).sum(-1))
        candidate_available_columns.append(available.any(-1))
        candidate_best_id_columns.append(ids[best_offsets])
    candidate_scores = torch.stack(candidate_best_columns, dim=-1)
    candidate_masses = torch.stack(candidate_mass_columns, dim=-1)
    candidate_available = torch.stack(candidate_available_columns, dim=-1)
    candidate_best_ids = torch.stack(candidate_best_id_columns, dim=-1)
    candidate_ranks_20 = 1 + (
        candidate_scores[:, None, :] > candidate_scores[:, :, None]
    ).sum(-1)
    nonnegative_scores = candidate_scores.clamp_min(0)
    candidate_denominator = nonnegative_scores.sum(-1, keepdim=True).clamp_min(1e-30)
    candidate_shares_20 = nonnegative_scores / candidate_denominator
    valid_vocab_sizes = (probabilities >= 0).sum(-1)

    full_vocab_ranks = None
    if include_full_vocab_ranks:
        full_vocab_ranks = torch.stack([
            (probabilities > candidate_scores[:, index:index + 1]).sum(-1) + 1
            for index in range(len(test_conditions))
        ], dim=-1)

    top_values = top_indices = None
    if include_top:
        top_values, top_indices = probabilities.topk(
            test_config["readout"]["saved_top_k"], dim=-1
        )

    arrays = {
        "scores": candidate_scores.detach().cpu(),
        "masses": candidate_masses.detach().cpu(),
        "available": candidate_available.detach().cpu(),
        "best_ids": candidate_best_ids.detach().cpu(),
        "ranks_20": candidate_ranks_20.detach().cpu(),
        "shares_20": candidate_shares_20.detach().cpu(),
        "valid_vocab_sizes": valid_vocab_sizes.detach().cpu(),
        "full_vocab_ranks": full_vocab_ranks.detach().cpu() if full_vocab_ranks is not None else None,
        "top_values": top_values.detach().cpu() if top_values is not None else None,
        "top_indices": top_indices.detach().cpu() if top_indices is not None else None,
    }
    summaries = []
    for row_index in range(probabilities.shape[0]):
        available_by_word = {
            word: bool(arrays["available"][row_index, index])
            for index, word in enumerate(test_conditions)
        }
        summary = {
            "valid_vocabulary_size": int(arrays["valid_vocab_sizes"][row_index]),
            "candidate_probabilities_json": json.dumps({
                word: float(arrays["scores"][row_index, index])
                for index, word in enumerate(test_conditions)
            }, sort_keys=True),
            "candidate_probability_masses_json": json.dumps({
                word: float(arrays["masses"][row_index, index])
                for index, word in enumerate(test_conditions)
            }, sort_keys=True),
            "candidate_available_json": json.dumps(available_by_word, sort_keys=True),
            "candidate_ranks_20_json": json.dumps({
                word: int(arrays["ranks_20"][row_index, index])
                if available_by_word[word] else None
                for index, word in enumerate(test_conditions)
            }, sort_keys=True),
            "candidate_probability_shares_20_json": json.dumps({
                word: float(arrays["shares_20"][row_index, index])
                if available_by_word[word] else None
                for index, word in enumerate(test_conditions)
            }, sort_keys=True),
            "candidate_best_token_ids_json": json.dumps({
                word: int(arrays["best_ids"][row_index, index])
                if available_by_word[word] else None
                for index, word in enumerate(test_conditions)
            }, sort_keys=True),
        }
        if include_full_vocab_ranks:
            ranks = {
                word: int(arrays["full_vocab_ranks"][row_index, index])
                if available_by_word[word] else None
                for index, word in enumerate(test_conditions)
            }
            valid_vocab = summary["valid_vocabulary_size"]
            summary.update({
                "candidate_full_vocab_ranks_json": json.dumps(ranks, sort_keys=True),
                "candidate_full_vocab_reciprocal_ranks_json": json.dumps({
                    word: (1.0 / rank if rank else None) for word, rank in ranks.items()
                }, sort_keys=True),
                "candidate_full_vocab_rank_percentiles_json": json.dumps({
                    word: (rank / valid_vocab if rank else None) for word, rank in ranks.items()
                }, sort_keys=True),
            })
        if include_top:
            top = [{
                "token_id": int(token_id),
                "token": tokenizer.decode([int(token_id)]),
                "probability": float(value),
            } for value, token_id in zip(
                arrays["top_values"][row_index], arrays["top_indices"][row_index]
            )]
            summary.update({
                "top1_token_id": top[0]["token_id"],
                "top1_token": top[0]["token"],
                "top1_probability": top[0]["probability"],
                "top5_token_ids_json": json.dumps([item["token_id"] for item in top[:5]]),
                "top10_json": json.dumps(top, ensure_ascii=False),
            })
        summaries.append(summary)
    return summaries

def summarize_base_candidate_distribution(probabilities):
    return summarize_base_candidate_batch(
        probabilities.unsqueeze(0), include_top=True, include_full_vocab_ranks=True
    )[0]


## One resumable base activation measurement

The forward pass is performed with all LoRA adapters disabled. Each prompt writes one response-average Parquet, one detailed-position Parquet, and then a `.done.json` marker through atomic replacement. Readout math and position selection mirror notebook 07; only target-specific fields are replaced by 20-candidate dictionaries.


In [ ]:
base_test_cells_dir = base_test_paths.lens_dir / "base_test_cells"
base_test_cells_dir.mkdir(parents=True, exist_ok=True)

def base_test_cell_paths(row):
    stem = f"{row['prompt_id']}__base"
    return (
        base_test_cells_dir / f"{stem}.aggregate.parquet",
        base_test_cells_dir / f"{stem}.positions.parquet",
        base_test_cells_dir / f"{stem}.done.json",
    )

def _measure_base_test_sequence_with_adapters_disabled(
    behavior_row, aggregate_path, positions_path, done_path
):
    prompt_ids = [int(token_id) for token_id in behavior_row["prompt_token_ids"]]
    generation_ids = [int(token_id) for token_id in behavior_row["generation_token_ids"]]
    complete_ids = prompt_ids + generation_ids
    assert generation_ids
    assert len(complete_ids) <= test_runtime["max_sequence_tokens"], len(complete_ids)
    prompt_length = len(prompt_ids)
    assistant_start = int(behavior_row["assistant_header_start"])
    input_start = max(0, min(
        assistant_start, prompt_length - test_config["readout"]["input_window"]
    ))
    response_stop = min(
        len(complete_ids), prompt_length + test_config["readout"]["response_position_limit"]
    )
    positions = list(range(input_start, response_stop))
    generated_positions = list(range(prompt_length, response_stop))
    generated_position_set = set(generated_positions)
    layers = list(lens.source_layers)
    emitted_token_ids = sorted(set(generation_ids))
    valid_emitted_ids = [
        token_id for token_id in emitted_token_ids if 0 <= token_id < base_test_vocabulary_size
    ]
    emitted_set = set(valid_emitted_ids)

    complete_tensor = torch.tensor([complete_ids], device=lens_model.input_device)
    with torch.no_grad(), ActivationRecorder(lens_model.layers, at=layers) as recorder:
        lens_model.forward(complete_tensor)

    common = {
        "schema_version": 1, "run_id": BASE_TEST_RUN_ID,
        "config_hash": base_control_config_hash,
        "source_adapter_test_run_id": TEST_RUN_ID,
        "prompt_id": behavior_row["prompt_id"],
        "prompt_type": behavior_row["prompt_type"], "split": behavior_row["split"],
        "paper_block_of_10": int(behavior_row["paper_block_of_10"]),
        "condition": "base", "target_word": None,
        "candidate_words_json": json.dumps(test_conditions),
        "emitted_token_ids_json": json.dumps(emitted_token_ids),
        "emitted_unique_token_count": len(emitted_token_ids),
        "generation_token_count": len(generation_ids),
        "any_candidate_mentioned": bool(behavior_row["any_candidate_mentioned"]),
        "base_model_revision": test_base_spec["revision"],
        "adapter_revision": None,
        "jlens_revision": test_config["jlens"]["revision"],
        "jlens_code_commit": test_config["jlens"]["official_code_commit"],
    }
    position_metadata = {
        position: test_position_metadata(complete_ids, prompt_length, assistant_start, position)
        for position in positions
    }
    aggregate_rows = []
    position_rows = []
    chunk_size = test_config["readout"]["position_chunk_size"]
    mask_protocols = [
        test_config["readout"]["primary_mask_protocol"],
        *test_config["readout"]["diagnostic_mask_protocols"],
    ]

    for layer in layers:
        source = recorder.activations[layer].detach()[0][positions].float()
        for method in test_config["readout"]["methods"]:
            residual = source if method == "logit_lens" else lens.transport(source, layer)
            response_probability_sums = {
                protocol: torch.zeros(
                    base_test_vocabulary_size, dtype=torch.float32, device=lens_model.input_device
                ) for protocol in mask_protocols
            }
            response_positions_counted = 0
            for chunk_start in range(0, len(positions), chunk_size):
                chunk_stop = min(len(positions), chunk_start + chunk_size)
                chunk_positions = positions[chunk_start:chunk_stop]
                chunk_residual = residual[chunk_start:chunk_stop]
                logits = lens_model.unembed(chunk_residual).float()
                probabilities = torch.softmax(logits, dim=-1)
                global_probabilities, global_logits = masked_probability_and_logits(
                    probabilities, logits, valid_emitted_ids
                )
                actual_masks = [
                    [int(complete_ids[position])] if position in generated_position_set else []
                    for position in chunk_positions
                ]
                position_probabilities, position_logits = masked_probability_and_logits(
                    probabilities, logits, actual_masks
                )
                unmasked_probabilities, unmasked_logits = masked_probability_and_logits(
                    probabilities, logits, None
                )
                for local_index, position in enumerate(chunk_positions):
                    if position in generated_position_set:
                        response_probability_sums["global_emitted_ids"] += global_probabilities[local_index].clamp_min(0)
                        response_probability_sums["position_actual_token"] += position_probabilities[local_index].clamp_min(0)
                        response_probability_sums["unmasked"] += unmasked_probabilities[local_index]
                        response_positions_counted += 1
                primary_summaries = summarize_base_candidate_batch(
                    global_probabilities, include_top=True, include_full_vocab_ranks=False
                )
                for row_index, position in enumerate(chunk_positions):
                    summary = primary_summaries[row_index]
                    top_ids = {item["token_id"] for item in json.loads(summary["top10_json"])}
                    assert not (top_ids & emitted_set), top_ids & emitted_set
                    position_rows.append({
                        **common, "method": method, "layer": int(layer),
                        "position": int(position), "mask_protocol": "global_emitted_ids",
                        **position_metadata[position], **summary,
                    })
                del primary_summaries
                del global_probabilities, global_logits, position_probabilities, position_logits
                del unmasked_probabilities, unmasked_logits, logits, probabilities

            assert response_positions_counted == len(generated_positions)
            for protocol in mask_protocols:
                average_probability = response_probability_sums[protocol] / response_positions_counted
                if protocol == "global_emitted_ids":
                    average_probability[valid_emitted_ids] = -1.0
                aggregate_summary = summarize_base_candidate_distribution(average_probability)
                if protocol == "global_emitted_ids":
                    aggregate_top_ids = {
                        item["token_id"] for item in json.loads(aggregate_summary["top10_json"])
                    }
                    assert not (aggregate_top_ids & emitted_set), aggregate_top_ids & emitted_set
                aggregate_rows.append({
                    **common, "method": method, "layer": int(layer),
                    "mask_protocol": protocol,
                    "aggregation": "mean_probability_over_generated_response_positions",
                    "response_positions_counted": response_positions_counted,
                    **aggregate_summary,
                })
                del average_probability
            del residual, response_probability_sums
        del source
        if layer % 12 == 0:
            torch.cuda.empty_cache()

    aggregate_frame = pd.DataFrame(aggregate_rows)
    position_frame = pd.DataFrame(position_rows)
    atomic_test_parquet(aggregate_frame, aggregate_path)
    atomic_test_parquet(position_frame, positions_path)
    done_payload = {
        "schema_version": 1, "completed_utc": utc_now(),
        "prompt_id": behavior_row["prompt_id"], "condition": "base",
        "aggregate_rows": len(aggregate_frame), "position_rows": len(position_frame),
        "aggregate_bytes": aggregate_path.stat().st_size,
        "position_bytes": positions_path.stat().st_size,
        "emitted_token_ids": emitted_token_ids,
    }
    done_tmp = done_path.with_suffix(done_path.suffix + ".tmp")
    done_tmp.write_text(json.dumps(done_payload, indent=2), encoding="utf-8")
    os.replace(done_tmp, done_path)
    del recorder, aggregate_frame, position_frame
    gc.collect(); torch.cuda.empty_cache()
    return done_payload

def measure_base_test_sequence(behavior_row, aggregate_path, positions_path, done_path):
    model.disable_adapters()
    try:
        return _measure_base_test_sequence_with_adapters_disabled(
            behavior_row, aggregate_path, positions_path, done_path
        )
    finally:
        model.enable_adapters()


## Run all 200 base sequences

The loop skips only complete atomic triples and prints a fresh ETA every ten prompts. It performs exactly one base-model activation pass per test prompt.


In [ ]:
ordered_base_behavior = base_behavior.assign(
    prompt_type_order=base_behavior["prompt_type"].map({"standard": 0, "direct": 1})
).sort_values(["prompt_type_order", "prompt_id"]).drop(columns="prompt_type_order")
pending_base_sequences = []
for row in ordered_base_behavior.to_dict("records"):
    aggregate_path, positions_path, done_path = base_test_cell_paths(row)
    if not (aggregate_path.exists() and positions_path.exists() and done_path.exists()):
        pending_base_sequences.append((row, aggregate_path, positions_path, done_path))
expected_base_sequences = 200
already_complete = expected_base_sequences - len(pending_base_sequences)
print(f"before base sweep: {already_complete}/{expected_base_sequences} complete")
base_sweep_started = time.time()
for new_index, (row, aggregate_path, positions_path, done_path) in enumerate(
    pending_base_sequences, start=1
):
    sequence_started = time.time()
    payload = measure_base_test_sequence(row, aggregate_path, positions_path, done_path)
    completed_total = already_complete + new_index
    if new_index == 1 or new_index % 10 == 0 or completed_total == expected_base_sequences:
        elapsed = time.time() - base_sweep_started
        rate = new_index / elapsed if elapsed > 0 else 0.0
        remaining = expected_base_sequences - completed_total
        eta_minutes = remaining / rate / 60 if rate > 0 else float("nan")
        print(
            f"base sweep {completed_total}/{expected_base_sequences} | "
            f"last={row['prompt_id']} {time.time() - sequence_started:.1f}s | "
            f"ETA {eta_minutes:.1f}m | position rows {payload['position_rows']}",
            flush=True,
        )


## Final integrity check

Notebook 08 should use the base control only after this cell confirms all 200 atomic triples and writes the completed pointer.


In [ ]:
base_done_files = sorted(base_test_cells_dir.glob("*.done.json"))
base_aggregate_files = sorted(base_test_cells_dir.glob("*.aggregate.parquet"))
base_position_files = sorted(base_test_cells_dir.glob("*.positions.parquet"))
assert len(base_done_files) == len(base_aggregate_files) == len(base_position_files) == 200
assert all(path.stat().st_size > 0 for path in base_aggregate_files + base_position_files)
base_completion = {
    "schema_version": 1, "completed_utc": utc_now(),
    "run_id": BASE_TEST_RUN_ID, "source_adapter_test_run_id": TEST_RUN_ID,
    "expected_sequences": 200, "completed_sequences": len(base_done_files),
    "aggregate_files": len(base_aggregate_files),
    "position_files": len(base_position_files),
    "aggregate_bytes": sum(path.stat().st_size for path in base_aggregate_files),
    "position_bytes": sum(path.stat().st_size for path in base_position_files),
    "methods": test_config["readout"]["methods"],
    "layers": [min(lens.source_layers), max(lens.source_layers)],
    "mask_protocols": [
        test_config["readout"]["primary_mask_protocol"],
        *test_config["readout"]["diagnostic_mask_protocols"],
    ],
    "candidate_words": list(test_conditions),
}
(base_test_paths.result_dir / "base_test_sweep_completion.json").write_text(
    json.dumps(base_completion, indent=2), encoding="utf-8"
)
update_manifest(base_test_paths, status="complete", **base_completion)
base_pointer_tmp = base_pointer_path.with_suffix(".json.tmp")
base_pointer_tmp.write_text(json.dumps({
    "run_id": BASE_TEST_RUN_ID,
    "source_adapter_test_run_id": TEST_RUN_ID,
    "config_hash": base_control_config_hash,
    "status": "complete", "updated_utc": utc_now(),
}, indent=2), encoding="utf-8")
os.replace(base_pointer_tmp, base_pointer_path)
model.enable_adapters()
display(base_completion)
